<a href="https://colab.research.google.com/github/Kintsukuro1/Mineria-Datos/blob/main/Canasta_Mercado_CoffeeShop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ☕ Análisis de Canasta de Mercado — Coffee Shop



In [8]:
!pip install mlxtend -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

pd.set_option('display.max_columns', None)

In [10]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]  # Toma el nombre real del archivo subido
print(f'Archivo cargado: {filename}')

df_raw = pd.read_excel(filename)
print(df_raw.shape)
df_raw.head()

Saving Coffee Shop Sales.xlsx to Coffee Shop Sales.xlsx
Archivo cargado: Coffee Shop Sales.xlsx
(149116, 11)


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [11]:
df = df_raw[['transaction_id', 'product_detail', 'product_id',
             'transaction_qty', 'unit_price']].copy()

df.columns = ['ID_boleta', 'Producto', 'Id_producto',
              'Cantidad', 'Precio_unitario']

df['Producto'] = df['Producto'].str.strip()
df.dropna(subset=['ID_boleta', 'Producto'], inplace=True)
df.head()

,ID_boleta,Producto,Id_producto,Cantidad,Precio_unitario
0,1,Ethiopia Rg,32,2,3.0
1,2,Spicy Eye Opener Chai Lg,57,2,3.1
2,3,Dark chocolate Lg,59,2,4.5
3,4,Our Old Time Diner Blend Sm,22,1,2.0
4,5,Spicy Eye Opener Chai Lg,57,2,3.1


Por qué: El dataset original tiene 11 columnas, solo necesitamos 5 para el modelo.
Para qué: Dejar el DataFrame limpio con los nombres acordados y sin filas vacías en columnas clave.
Resultado: Tabla con exactamente las 5 columnas renombradas y sin valores nulos.

In [12]:
basket_series = df.groupby('ID_boleta')['Producto'].apply(list)

te = TransactionEncoder()
te_array = te.fit_transform(basket_series)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(df_encoded.shape)
df_encoded.head(3)

(149116, 80)


,Almond Croissant,Brazilian - Organic,Brazilian Lg,Brazilian Rg,Brazilian Sm,Cappuccino,Cappuccino Lg,Carmel syrup,Chili Mayan,Chocolate Chip Biscotti,Chocolate Croissant,Chocolate syrup,Civet Cat,Columbian Medium Roast,Columbian Medium Roast Lg,Columbian Medium Roast Rg,Columbian Medium Roast Sm,Cranberry Scone,Croissant,Dark chocolate,Dark chocolate Lg,Dark chocolate Rg,Earl Grey,Earl Grey Lg,Earl Grey Rg,English Breakfast,English Breakfast Lg,English Breakfast Rg,Espresso Roast,Espresso shot,Ethiopia,Ethiopia Lg,Ethiopia Rg,Ethiopia Sm,Ginger Biscotti,Ginger Scone,Guatemalan Sustainably Grown,Hazelnut Biscotti,Hazelnut syrup,I Need My Bean! Diner mug,I Need My Bean! Latte cup,I Need My Bean! T-shirt,Jamacian Coffee River,Jamaican Coffee River Lg,Jamaican Coffee River Rg,Jamaican Coffee River Sm,Jumbo Savory Scone,Latte,Latte Rg,Lemon Grass,Lemon Grass Lg,Lemon Grass Rg,Morning Sunrise Chai,Morning Sunrise Chai Lg,Morning Sunrise Chai Rg,Oatmeal Scone,Organic Decaf Blend,Our Old Time Diner Blend,Our Old Time Diner Blend Lg,Our Old Time Diner Blend Rg,Our Old Time Diner Blend Sm,Ouro Brasileiro shot,Peppermint,Peppermint Lg,Peppermint Rg,Primo Espresso Roast,Scottish Cream Scone,Serenity Green Tea,Serenity Green Tea Lg,Serenity Green Tea Rg,Spicy Eye Opener Chai,Spicy Eye Opener Chai Lg,Spicy Eye Opener Chai Rg,Sugar Free Vanilla syrup,Sustainably Grown Organic,Sustainably Grown Organic Lg,Sustainably Grown Organic Rg,Traditional Blend Chai,Traditional Blend Chai Lg,Traditional Blend Chai Rg
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


Por qué: FP-Growth no trabaja con filas de productos, necesita saber qué productos estuvieron juntos en cada boleta.
Para qué: Transformar los datos a una matriz binaria donde cada fila es una boleta y cada columna es un producto con True/False.
Resultado: Matriz de ~149.000 boletas × 80 productos con valores True o False.

In [13]:
MIN_SUPPORT    = 0.01
MIN_CONFIDENCE = 0.10
MIN_LIFT       = 1.0

frequent_itemsets = fpgrowth(df_encoded, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print(f'Itemsets encontrados: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False).head(10)

Itemsets encontrados: 56


,support,itemsets,length
21,0.020628,(Chocolate Croissant),1
42,0.020474,(Earl Grey Rg),1
2,0.020313,(Dark chocolate Lg),1
26,0.020293,(Morning Sunrise Chai Rg),1
25,0.020206,(Columbian Medium Roast Rg),1
22,0.020052,(Latte),1
15,0.019857,(Sustainably Grown Organic Lg),1
50,0.019817,(Traditional Blend Chai Rg),1
1,0.019790,(Spicy Eye Opener Chai Lg),1
31,0.019777,(Peppermint Rg),1


Por qué: El algoritmo necesita un umbral mínimo para no generar miles de combinaciones irrelevantes.
Para qué: Encontrar todos los grupos de productos que aparecen juntos en al menos el 1% de las boletas.
Resultado: Lista de combinaciones frecuentes ordenadas por support (qué tan seguido aparecen juntas).

In [14]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=MIN_LIFT)
rules = rules[rules['confidence'] >= MIN_CONFIDENCE]

rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)

,antecedents,consequents,support,confidence,lift


Por qué: Los itemsets solos no dicen la dirección de la relación, las reglas sí: "si compra A → compra B".
Para qué: Convertir las combinaciones frecuentes en reglas concretas con métricas de calidad.
Resultado: Tabla con las 15 mejores reglas mostrando qué producto lleva a qué otro, con su confianza y lift.

In [15]:
for _, row in rules.head(5).iterrows():
    print(f"Si compra: '{row['antecedents']}'")
    print(f"→ Compra:  '{row['consequents']}'")
    print(f"   Confidence: {row['confidence']:.2%} | Lift: {row['lift']:.2f}\n")

Por qué: Las tablas con números son difíciles de interpretar de un vistazo.
Para qué: Presentar las reglas más importantes en formato legible para tomar decisiones de negocio.
Resultado: Las 5 mejores reglas impresas como frases, con su porcentaje de confianza y lift.

In [16]:
def recomendar(producto, top_n=5):
    mask = rules['antecedents'].str.contains(producto, case=False, na=False)
    resultado = rules[mask][['antecedents', 'consequents', 'confidence', 'lift']]
    resultado = resultado.sort_values('lift', ascending=False).head(top_n)
    for _, row in resultado.iterrows():
        print(f"→ {row['consequents']}")
        print(f"   Confidence: {row['confidence']:.1%} | Lift: {row['lift']:.2f}")

recomendar('Latte Rg')  # Cambia por el producto que quieras consultar

Por qué: Las reglas en tabla son útiles para análisis, pero no para consultas rápidas por producto.
Para qué: Tener una función reutilizable que, dado cualquier producto, devuelve qué otros se recomiendan junto a él.
Resultado: Lista de productos recomendados para "Latte Rg", ordenados por lift. Cambia el nombre por cualquier producto del catálogo.